
# Medical Image Segmentation using U-Net

This cleaned notebook reproduces the supplied experiment without overstating the dataset. It generates **synthetic MRI-style grayscale images with elliptical binary masks**, trains a compact U-Net, compares it with an intensity-threshold baseline, and saves evaluation artifacts.

> **Medical disclaimer:** Educational portfolio demonstration only. This is not a medical diagnostic tool and the dataset contains no real patient scans.


In [ ]:

from pathlib import Path
import json
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import MODEL_PATH, OUTPUT_DIR, SEED
from src.metrics import dice_coefficient_np, iou_score_np
from src.model_evaluation import intensity_threshold_baseline, evaluate_predictions, predict_probabilities
from src.model_training import TrainingConfig, save_training_artifacts, train_model
from src.synthetic_data import generate_synthetic_dataset, split_dataset

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


## 1. Generate and inspect the deterministic synthetic dataset

In [ ]:

X, Y = generate_synthetic_dataset(num_samples=2500, seed=SEED)
X_train, X_val, X_test, y_train, y_val, y_test = split_dataset(X, Y, seed=SEED)
print("Full:", X.shape, Y.shape)
print("Train / validation / test:", len(X_train), len(X_val), len(X_test))
print("Training positive-pixel rate:", float(y_train.mean()))


In [ ]:

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for idx in range(4):
    axes[0, idx].imshow(X_train[idx].squeeze(), cmap="gray")
    axes[0, idx].axis("off")
    axes[1, idx].imshow(y_train[idx].squeeze(), cmap="gray")
    axes[1, idx].axis("off")
axes[0, 0].set_title("Synthetic image")
axes[1, 0].set_title("Binary mask")
plt.tight_layout()


## 2. Intensity-threshold baseline

In [ ]:

baseline = intensity_threshold_baseline(X_test, threshold=0.55)
baseline_metrics = {
    "dice": dice_coefficient_np(y_test, baseline),
    "iou": iou_score_np(y_test, baseline),
}
baseline_metrics


## 3. Train the compact U-Net

In [ ]:

config = TrainingConfig(epochs=15, batch_size=32, learning_rate=0.001)
model, history = train_model(X_train, y_train, X_val, y_val, config=config)
model.summary()


In [ ]:

history_df = pd.DataFrame(history.history)
history_df[["dice_coef_tf", "val_dice_coef_tf"]].plot(figsize=(9, 4), title="Dice history")
plt.show()
history_df[["loss", "val_loss"]].plot(figsize=(9, 4), title="Loss history")
plt.show()


## 4. Evaluate and save

In [ ]:

probabilities = predict_probabilities(model, X_test)
unet_metrics = evaluate_predictions(y_test, probabilities, threshold=0.5)
comparison = pd.DataFrame([
    {"approach": "Intensity threshold", **baseline_metrics},
    {"approach": "U-Net", "dice": unet_metrics["soft_dice"], "iou": unet_metrics["soft_iou"]},
])
comparison


In [ ]:

save_training_artifacts(model, history, model_path=MODEL_PATH, output_dir=OUTPUT_DIR, config=config)
(OUTPUT_DIR / "retrained_metrics.json").write_text(json.dumps(unet_metrics, indent=2), encoding="utf-8")
print("Saved model:", MODEL_PATH)



## Interpretation and limitations

The U-Net outperforms the threshold baseline on this synthetic distribution. The scores are unusually high because the target is deliberately brighter than the background and has a simple elliptical geometry. A real medical-imaging study would require licensed and de-identified data, patient-level splitting, modality-specific preprocessing, qualified annotation, external validation, and clinical review.
